In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import json
import io
import time
import os
from dotenv import load_dotenv

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv('FINRA_TOKEN')
headers = {
    'Authorization': f'Bearer {token}',
    'Content-Type': 'application/json',
    'Accept': 'application/json'
}

base_url = 'https://api.finra.org/data/group/otcMarket/name/weeklySummary'

In [3]:
# Tune these as needed
LIMIT      = 5000   # max records per request (FINRA sync cap is 5000)
SLEEP      = 0.5    # seconds between requests to avoid rate limiting
MAX_RETRY  = 3      # retries per page before giving up
RETRY_WAIT = 5      # seconds to wait before each retry

# Month chunks to iterate over – add/remove months as data becomes available
MONTHS_2026 = [
    ('2026-01-01', '2026-01-31'),
    ('2026-02-01', '2026-02-28'),
    ('2026-03-01', '2026-03-31'),
    ('2026-04-01', '2026-04-30'),
    ('2026-05-01', '2026-05-31'),
    ('2026-06-01', '2026-06-30'),
    ('2026-07-01', '2026-07-31'),
    ('2026-08-01', '2026-08-31'),
    ('2026-09-01', '2026-09-30'),
    ('2026-10-01', '2026-10-31'),
    ('2026-11-01', '2026-11-30'),
    ('2026-12-01', '2026-12-31'),
]

In [4]:
def fetch_month(start_date: str, end_date: str) -> list:
    """Fetch all pages for a single month window, with retry on connection errors."""
    month_data = []
    offset = 0

    while True:
        payload = {
            "limit": LIMIT,
            "offset": offset,
            "dateRangeFilters": [
                {
                    "fieldName": "weekStartDate",
                    "startDate": start_date,
                    "endDate": end_date
                }
            ]
        }

        # Retry loop for transient connection resets
        for attempt in range(1, MAX_RETRY + 1):
            try:
                response = requests.post(
                    base_url, json=payload, headers=headers,
                    timeout=60          # explicit timeout so we don't hang forever
                )
                response.raise_for_status()
                data = response.json()
                break                   # success – exit retry loop

            except requests.exceptions.HTTPError as e:
                print(f"  HTTP error (attempt {attempt}/{MAX_RETRY}): {e}")
                print(f"  Response: {response.text[:300]}")
                if attempt == MAX_RETRY:
                    return month_data   # return whatever we have so far
                time.sleep(RETRY_WAIT)

            except requests.exceptions.RequestException as e:
                print(f"  Connection error (attempt {attempt}/{MAX_RETRY}): {e}")
                if attempt == MAX_RETRY:
                    return month_data
                time.sleep(RETRY_WAIT * attempt)   # back off longer each retry

        if not data:                    # empty page → done with this month
            break

        month_data.extend(data)
        print(f"    offset={offset:>7,}  page={len(data):>5,}  month total={len(month_data):>6,}")

        if len(data) < LIMIT:           # partial page → last page for this month
            break

        offset += LIMIT
        time.sleep(SLEEP)

    return month_data


def fetch_all_finra_data() -> pd.DataFrame:
    """Iterate month-by-month across 2026 to stay well under FINRA's offset limit.

    Chunking by month keeps each window small enough that offsets never
    approach the 500k ceiling, and a single dropped connection only loses
    one page rather than the entire run.
    """
    all_data = []

    for start, end in MONTHS_2026:
        print(f"\nFetching {start} → {end} …")
        month_records = fetch_month(start, end)
        all_data.extend(month_records)
        print(f"  Month done: {len(month_records):,} records  |  Grand total: {len(all_data):,}")
        time.sleep(SLEEP)               # brief pause between months

    return pd.DataFrame(all_data)

In [5]:
# Check for existing cache and only fetch new data
CACHE_FILE = "finra_weekly_summary_2026.pkl"

# Load existing cache if available
existing_df = None
if os.path.exists(CACHE_FILE):
    try:
        existing_df = pd.read_pickle(CACHE_FILE)
        print(f"Loaded existing cache with {len(existing_df):,} records")
    except Exception as e:
        print(f"Could not load cache: {e}")
        existing_df = None

# Determine which week dates we already have (data is weekly, not daily)
existing_weeks = set()
if existing_df is not None and 'weekStartDate' in existing_df.columns:
    existing_df['weekStartDate'] = pd.to_datetime(existing_df['weekStartDate'])
    existing_weeks = set(existing_df['weekStartDate'].dt.date.unique())
    print(f"Existing cache covers {len(existing_weeks)} unique week dates")

# Filter months to only fetch new data
new_months = []
for start, end in MONTHS_2026:
    start_dt = datetime.strptime(start, '%Y-%m-%d').date()
    end_dt = datetime.strptime(end, '%Y-%m-%d').date()
    # Check if we have any weeks in this month
    # Get all weeks that fall within this month (check each week start)
    month_weeks = set()
    current = start_dt
    while current <= end_dt:
        month_weeks.add(current)
        current += timedelta(weeks=1)
    
    if not month_weeks.issubset(existing_weeks):
        new_months.append((start, end))
        print(f"  Will fetch new data for {start} → {end}")
    else:
        print(f"  Skipping {start} → {end} (all weeks cached)")

if new_months:
    print(f"\nFetching {len(new_months)} new months...")
    all_new_data = []
    for start, end in new_months:
        print(f"\nFetching {start} → {end} …")
        month_records = fetch_month(start, end)
        all_new_data.extend(month_records)
        print(f"  Month done: {len(month_records):,} records")
        time.sleep(SLEEP)
    
    if all_new_data:
        new_df = pd.DataFrame(all_new_data)
        if 'weekStartDate' in new_df.columns:
            new_df['weekStartDate'] = pd.to_datetime(new_df['weekStartDate'])
        
        # Merge with existing cache
        if existing_df is not None:
            df = pd.concat([existing_df, new_df], ignore_index=True)
            df = df.drop_duplicates(subset=df.columns.tolist(), keep='last')
            print(f"\nMerged: {len(existing_df):,} cached + {len(new_df):,} new = {len(df):,} total")
        else:
            df = new_df
            print(f"\nTotal new records: {len(df)}")
else:
    print("\nAll data already cached, using existing data")
    df = existing_df if existing_df is not None else df

# Quick sanity check
if 'weekStartDate' in df.columns:
    df['weekStartDate'] = pd.to_datetime(df['weekStartDate'])
    print(f"Date range in results: {df['weekStartDate'].min()} → {df['weekStartDate'].max()}")

# Save to disk
df.to_pickle(CACHE_FILE)
print(f"Saved {len(df):,} records to {CACHE_FILE}")

  Will fetch new data for 2026-01-01 → 2026-01-31
  Will fetch new data for 2026-02-01 → 2026-02-28
  Will fetch new data for 2026-03-01 → 2026-03-31
  Will fetch new data for 2026-04-01 → 2026-04-30
  Will fetch new data for 2026-05-01 → 2026-05-31
  Will fetch new data for 2026-06-01 → 2026-06-30
  Will fetch new data for 2026-07-01 → 2026-07-31
  Will fetch new data for 2026-08-01 → 2026-08-31
  Will fetch new data for 2026-09-01 → 2026-09-30
  Will fetch new data for 2026-10-01 → 2026-10-31
  Will fetch new data for 2026-11-01 → 2026-11-30
  Will fetch new data for 2026-12-01 → 2026-12-31

Fetching 12 new months...

Fetching 2026-01-01 → 2026-01-31 …
    offset=      0  page=5,000  month total= 5,000
    offset=  5,000  page=5,000  month total=10,000
    offset= 10,000  page=5,000  month total=15,000
    offset= 15,000  page=5,000  month total=20,000
    offset= 20,000  page=5,000  month total=25,000
    offset= 25,000  page=5,000  month total=30,000
    offset= 30,000  page=5,000 